# 01 — Data exploration (Phase 1)

**Goal:** understand what the files look like *before* doing any ML.

We will:
1. Load each TSV file **one at a time** (the laptop has ~8 GB RAM; all files together would not fit comfortably).
2. Print basic facts: shape, columns, missing values, duplicate IDs, countries, string lengths.
3. Load the ground truth and see how Source 1 connects to Source 2 / Source 3.

Nothing in `dataset/` is modified.

In [1]:
import sys, gc, io
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_loading import read_tsv, load_source, load_ground_truth, ground_truth_to_pairs

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Why `sep="\t"`?

TSV = **Tab**-Separated Values. Each column is separated by an invisible TAB character.
Addresses contain commas (`"85 Wayne Avenue, Ticonderoga, NY"`), so if we split on commas,
one address would be broken into several fake columns. The tiny example below shows the difference.

In [2]:
one_line = "entity_id\tbusiness_name\tbusiness_address\tcountry\nS1-1\tMaure Williams Inc\t85 Wayne Avenue, Ticonderoga, NY\tUS\n"

print("Split on TAB (correct):")
display(pd.read_csv(io.StringIO(one_line), sep="\t"))

print("Split on COMMA (wrong) - the address is chopped up and columns are garbage:")
display(pd.read_csv(io.StringIO(one_line), sep=","))

Split on TAB (correct):


,entity_id,business_name,business_address,country
0,S1-1,Maure Williams Inc,"85 Wayne Avenue, Ticonderoga, NY",US


Split on COMMA (wrong) - the address is chopped up and columns are garbage:


,,entity_id\tbusiness_name\tbusiness_address\tcountry
S1-1\tMaure Williams Inc\t85 Wayne Avenue,Ticonderoga,NY\tUS


## A reusable "profile" function

For each source file we want the same facts, so we write them once in a function.

- **Missing values:** we count *empty strings* `""`, because we load with `keep_default_na=False`
  (empty stays empty, and text like `"null"` is kept as text instead of silently becoming NaN).
- **Duplicate IDs:** every `entity_id` should be unique. If not, the output files would be ambiguous.
- **Lengths:** number of characters in names and addresses — tells us how much text we have to compare.

In [3]:
summary_rows = []

def profile_source(df, name):
    print(f"==================== {name} ====================")
    print("shape (rows, columns):", df.shape)
    display(df.head(5))

    print("Memory used by this table: %.0f MB" % (df.memory_usage(deep=True).sum() / 1e6))
    print("\nEmpty values per column:")
    print((df == "").sum().to_string())

    print("\nDuplicate entity_ids:", df["entity_id"].duplicated().sum())
    print("ID prefixes:", df["entity_id"].str[:3].value_counts().to_dict())

    print("\nCountry counts:")
    print(df["country"].value_counts().to_string())

    name_len = df["business_name"].str.len()
    addr_len = df["business_address"].str.len()
    print("\nName length (characters):")
    print(name_len.describe().round(1).to_string())
    print("\nAddress length (characters):")
    print(addr_len.describe().round(1).to_string())

    n_unique_names = df["business_name"].nunique()
    n_unique_addr = df["business_address"].nunique()
    print(f"\nUnique business names: {n_unique_names:,} of {len(df):,}")
    print(f"Unique addresses:      {n_unique_addr:,} of {len(df):,}")

    summary_rows.append({
        "file": name,
        "rows": len(df),
        "duplicate_ids": int(df["entity_id"].duplicated().sum()),
        "empty_names": int((df["business_name"] == "").sum()),
        "empty_addresses": int((df["business_address"] == "").sum()),
        "countries": ", ".join(sorted(df["country"].unique())),
        "median_name_len": float(name_len.median()),
        "median_addr_len": float(addr_len.median()),
        "unique_names": n_unique_names,
        "unique_addresses": n_unique_addr,
    })

## Profile every source file (one at a time)

Load → profile → delete → free memory. Each file takes ~15 s to load.

In [4]:
df = load_source("train", 1)
profile_source(df, "train_source1")
del df; gc.collect();

==================== train_source1 ====================
shape (rows, columns): (2206821, 4)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India


Memory used by this table: 272 MB

Empty values per column:
entity_id           0
business_name       0
business_address    0
country             0



Duplicate entity_ids: 0
ID prefixes: {'S1-': 2206821}

Country counts:
country
US       1323633
India     883188



Name length (characters):
count    2206821.0
mean          24.0
std            7.7
min            3.0
25%           18.0
50%           24.0
75%           30.0
max          105.0

Address length (characters):
count    2206821.0
mean          52.1
std           25.3
min           11.0
25%           33.0
50%           41.0
75%           70.0
max          256.0



Unique business names: 1,539,229 of 2,206,821
Unique addresses:      2,130,606 of 2,206,821


In [5]:
df = load_source("train", 2)
profile_source(df, "train_source2")
del df; gc.collect();

==================== train_source2 ====================
shape (rows, columns): (5034616, 4)


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


Memory used by this table: 630 MB

Empty values per column:
entity_id                0
business_name            0
business_address    168967
country                  0



Duplicate entity_ids: 0
ID prefixes: {'S2-': 5034616}

Country counts:


country
US       3016817
India    2017799



Name length (characters):
count    5034616.0
mean          25.1
std            8.9
min            2.0
25%           19.0
50%           25.0
75%           31.0
max          104.0

Address length (characters):


count    5034616.0
mean          46.2
std           24.8
min            0.0
25%           30.0
50%           37.0
75%           61.0
max          249.0



Unique business names: 4,402,009 of 5,034,616
Unique addresses:      4,337,262 of 5,034,616


In [6]:
df = load_source("train", 3)
profile_source(df, "train_source3")
del df; gc.collect();

==================== train_source3 ====================
shape (rows, columns): (5285603, 4)


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block Jayanagar, Bengaluru Urban, Ban...",India


Memory used by this table: 652 MB

Empty values per column:
entity_id                0
business_name            0
business_address    175916
country                  0



Duplicate entity_ids: 0
ID prefixes: {'S3-': 5285603}

Country counts:


country
US       3170056
India    2115547



Name length (characters):
count    5285603.0
mean          25.2
std            9.5
min            2.0
25%           18.0
50%           25.0
75%           31.0
max          123.0

Address length (characters):


count    5285603.0
mean          46.7
std           21.6
min            0.0
25%           35.0
50%           42.0
75%           54.0
max          240.0



Unique business names: 4,651,609 of 5,285,603
Unique addresses:      4,632,765 of 5,285,603


In [7]:
df = load_source("test", 1)
profile_source(df, "test_source1")
del df; gc.collect();

==================== test_source1 ====================
shape (rows, columns): (1732544, 4)


,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine",France
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar, Awas Vikas Colony",India
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Rue Pierre Dignac",France


Memory used by this table: 224 MB

Empty values per column:
entity_id           0
business_name       0
business_address    0
country             0



Duplicate entity_ids: 0
ID prefixes: {'S1-': 1732544}

Country counts:
country
India     809986
US        663106
France    259452



Name length (characters):
count    1732544.0
mean          23.8
std            7.7
min            3.0
25%           18.0
50%           24.0
75%           29.0
max           92.0

Address length (characters):
count    1732544.0
mean          57.2
std           25.0
min           11.0
25%           36.0
50%           50.0
75%           74.0
max          268.0



Unique business names: 1,238,867 of 1,732,544
Unique addresses:      1,677,483 of 1,732,544


In [8]:
df = load_source("test", 2)
profile_source(df, "test_source2")
del df; gc.collect();

==================== test_source2 ====================
shape (rows, columns): (4887273, 4)


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karnataka",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US


Memory used by this table: 646 MB

Empty values per column:
entity_id                0
business_name            0
business_address    129408
country                  0



Duplicate entity_ids: 0
ID prefixes: {'S2-': 4887273}

Country counts:


country
India     2312565
US        1871330
France     703378



Name length (characters):
count    4887273.0
mean          25.7
std            9.1
min            2.0
25%           19.0
50%           25.0
75%           32.0
max          102.0

Address length (characters):


count    4887273.0
mean          50.4
std           25.3
min            0.0
25%           32.0
50%           43.0
75%           67.0
max          269.0



Unique business names: 4,311,041 of 4,887,273
Unique addresses:      4,224,784 of 4,887,273


In [9]:
df = load_source("test", 3)
profile_source(df, "test_source3")
del df; gc.collect();

==================== test_source3 ====================
shape (rows, columns): (5082316, 4)


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V, Gurugram, Gurgaon, HR",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, null, A-68, दिल्ली",India


Memory used by this table: 648 MB

Empty values per column:
entity_id                0
business_name            0
business_address    136098
country                  0



Duplicate entity_ids: 0


ID prefixes: {'S3-': 5082316}

Country counts:
country
India     2405000
US        1945701
France     731615



Name length (characters):
count    5082316.0
mean          25.7
std            9.6
min            2.0
25%           19.0
50%           25.0
75%           32.0
max          103.0

Address length (characters):


count    5082316.0
mean          48.7
std           22.6
min            0.0
25%           35.0
50%           43.0
75%           59.0
max          267.0



Unique business names: 4,521,929 of 5,082,316
Unique addresses:      4,456,436 of 5,082,316


In [10]:
summary = pd.DataFrame(summary_rows).set_index("file")
summary

,rows,duplicate_ids,empty_names,empty_addresses,countries,median_name_len,median_addr_len,unique_names,unique_addresses
file,,,,,,,,,
train_source1,2206821,0,0,0,"India, US",24.0,41.0,1539229,2130606
train_source2,5034616,0,0,168967,"India, US",25.0,37.0,4402009,4337262
train_source3,5285603,0,0,175916,"India, US",25.0,42.0,4651609,4632765
test_source1,1732544,0,0,0,"France, India, US",24.0,50.0,1238867,1677483
test_source2,4887273,0,0,129408,"France, India, US",25.0,43.0,4311041,4224784
test_source3,5082316,0,0,136098,"France, India, US",25.0,43.0,4521929,4456436


### What to notice
- **Test has France**, train has only US and India. Our code must treat `country` as an open-ended text field.
- Source 1 addresses are never empty; Source 2 / 3 sometimes are.
- Unique names are fewer than rows → several different records share the same name
  (different branches, or different businesses with the same name). Name alone is not enough.

## Ground truth (labels) — only exists for train

`train_ground_truth.tsv` has **one row per Source 1 entity**:

| source1_entity_id | matched_entity_ids |
|---|---|
| S1-965667 | S2-681193310,S2-743505751,S3-775321672,... |
| S1-... | *(empty = this business has no match)* |

We also convert it to a **long** table (one row per matching pair), which is easier to count and join.

In [11]:
gt = load_ground_truth()
print("shape:", gt.shape)
display(gt.head())

pairs = ground_truth_to_pairs(gt)
print("Number of true (S1, match) pairs:", f"{len(pairs):,}")
print("Distinct matched S2/S3 ids:       ", f"{pairs['match_id'].nunique():,}")
print("S2/S3 ids matched to more than one S1:", pairs["match_id"].duplicated().sum())

shape: (2206821, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785"


Number of true (S1, match) pairs: 7,638,365


Distinct matched S2/S3 ids:        7,638,365


S2/S3 ids matched to more than one S1: 0


In [12]:
matches_per_s1 = pairs.groupby("source1_entity_id").size().reindex(gt["source1_entity_id"], fill_value=0)
s2_per_s1 = pairs[pairs["match_source"] == "S2"].groupby("source1_entity_id").size().reindex(gt["source1_entity_id"], fill_value=0)
s3_per_s1 = pairs[pairs["match_source"] == "S3"].groupby("source1_entity_id").size().reindex(gt["source1_entity_id"], fill_value=0)

distribution = pd.DataFrame({
    "total matches": matches_per_s1.value_counts().sort_index(),
    "from S2": s2_per_s1.value_counts().sort_index(),
    "from S3": s3_per_s1.value_counts().sort_index(),
}).fillna(0).astype(int)
distribution.index.name = "number of matches"
print(f"Source 1 entities with NO match: {(matches_per_s1 == 0).sum():,} ({(matches_per_s1 == 0).mean():.1%})")
distribution

Source 1 entities with NO match: 123,247 (5.6%)


,total matches,from S2,from S3
number of matches,,,
0,123247,287745,266276
1,119157,789108,716417
2,375212,652779,668375
3,530841,333957,372443
4,484115,119078,145116
5,321957,24154,35378
6,164868,0,2816
7,63968,0,0
8,18680,0,0


## How the records connect — real examples side by side

We pick a few Source 1 entities, look up their matched IDs in the ground truth,
then fetch those records from Source 2 and Source 3.

To save memory we load each source, keep only the rows we need, and delete the rest.

In [13]:
example_ids = (
    list(gt[gt["matched_entity_ids"] != ""].sample(3, random_state=42)["source1_entity_id"])
    + ["S1-55344266"]                                                          # Tamil-script example
    + list(gt[gt["matched_entity_ids"] == ""].sample(1, random_state=42)["source1_entity_id"])  # no match
)
wanted_match_ids = set(pairs.loc[pairs["source1_entity_id"].isin(example_ids), "match_id"])

s1_rows = load_source("train", 1)
s1_rows = s1_rows[s1_rows["entity_id"].isin(example_ids)]; gc.collect()

match_rows = []
for source_number in [2, 3]:
    df = load_source("train", source_number)
    match_rows.append(df[df["entity_id"].isin(wanted_match_ids)])
    del df; gc.collect()
match_rows = pd.concat(match_rows)

In [14]:
for s1_id in example_ids:
    s1 = s1_rows[s1_rows["entity_id"] == s1_id]
    ids = pairs.loc[pairs["source1_entity_id"] == s1_id, "match_id"]
    print(f"\n########## {s1_id}  ->  {len(ids)} match(es): {', '.join(ids) if len(ids) else '(none)'}")
    display(pd.concat([s1, match_rows[match_rows["entity_id"].isin(ids)]]).reset_index(drop=True))


########## S1-777210964  ->  4 match(es): S2-12029274, S2-407524980, S3-970714334, S3-712922821


,entity_id,business_name,business_address,country
0,S1-777210964,Davis Family Office,"88 Olive Circle, Lebanon, TN",US
1,S2-407524980,DAVIS FAMILY OFFICE,"OLIVE CIR, LEBANON, TN",US
2,S2-12029274,Davis Family Offie,"88 OLIVE CIR, LEBANON, TN",US
3,S3-970714334,Davis Family (Office),"88 Olive Cir, Lebanon, Tennessee",US
4,S3-712922821,Davis Family,"Lebanon, Tennessee, 88 Olive Cir",US



########## S1-190133982  ->  3 match(es): S2-523219173, S3-716445195, S3-986804760


,entity_id,business_name,business_address,country
0,S1-190133982,MS Consultancy Corp,"Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pun...",India
1,S2-523219173,MS [Consultancy],"SHYMALA APPTS 1-1ST FLOOR FLAT NO. 4 OPP RATNA HOSPITAL SB ROAD IN HAVELI, P...",India
2,S3-716445195,M5 Consultancy Corp,"Pune, MH, Pune, Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Roa...",India
3,S3-986804760,MS Corp Services,"Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pun...",India



########## S1-970131671  ->  2 match(es): S2-670035601, S3-392756625


,entity_id,business_name,business_address,country
0,S1-970131671,3520 Main Road Realty Inc,"TX, 158 Simpson Lane, Somerset",US
1,S2-670035601,3520 MAIN ROAD REALTY INC,"SIMPSON LANE, SOMERSET, TX",US
2,S3-392756625,3520 MAIN ROAD Realty INC,"158 Simpson Lane, Somerset, Texas",US



########## S1-55344266  ->  4 match(es): S2-249013014, S2-197070651, S3-478195123, S3-384364074


,entity_id,business_name,business_address,country
0,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",India
1,S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
2,S2-249013014,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu",India
3,S3-384364074,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு",India
4,S3-478195123,Raj Investments எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN",India



########## S1-303396463  ->  0 match(es): (none)


,entity_id,business_name,business_address,country
0,S1-303396463,Indo Super Impex Private Limited,"242 - Ground Floor, Mumbai, Geeta Nagar, Dr Homi Bhabha Road, Navy Nagar, Co...",India


### What to notice in these examples
- **Typos:** "Williams" vs "Wilblims", "Wayne" vs "Wanye".
- **Different scripts:** the same Indian business written in English and in Tamil/Hindi script.
- **Missing addresses** in some Source 2/3 records.
- **Reordered addresses:** "Kansas City, MO, 630 45ND TERRACE" vs "630 45th Terrace, Kansas City, MO".
- **Sometimes the name is totally different** and only the address connects the records.
- **Some Source 1 entities have no match at all** — our system must be allowed to answer "none".